# 05 — Advanced Models: XGBoost, LightGBM, CatBoost
**Owner:** Panashe  |  **Phase:** 4  |  **Date:** May 13-14

Objectives: train 3 gradient boosting models, save OOF + test predictions for ensemble.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.preprocessing import load_processed
from src.models import get_lgbm, get_xgb, get_catboost
from src.train import load_folds, cross_validate_model, train_full
from src.evaluate import compute_metrics, plot_roc_curve, plot_feature_importance
from src.config import TARGET_COL, ID_COL
import pathlib
pathlib.Path("models").mkdir(exist_ok=True)

train_proc, test_proc = load_processed()
FEATURE_COLS = [c for c in train_proc.columns if c not in [TARGET_COL, ID_COL]]
X = train_proc[FEATURE_COLS].values
y = train_proc[TARGET_COL].values
X_test = test_proc[FEATURE_COLS].values
folds = load_folds()

## 1. LightGBM

In [ ]:
lgbm_results = cross_validate_model(get_lgbm(), X, y, folds)
np.save("models/lgbm_oof.npy", lgbm_results["oof_preds"])
lgbm_full = train_full(get_lgbm(), X, y)
np.save("models/lgbm_test.npy", lgbm_full.predict_proba(X_test)[:,1])
print(f"LightGBM OOF AUC: {lgbm_results['oof_auc']:.5f}")

## 2. XGBoost

In [ ]:
xgb_results = cross_validate_model(get_xgb(), X, y, folds)
np.save("models/xgb_oof.npy", xgb_results["oof_preds"])
xgb_full = train_full(get_xgb(), X, y)
np.save("models/xgb_test.npy", xgb_full.predict_proba(X_test)[:,1])
print(f"XGBoost OOF AUC: {xgb_results['oof_auc']:.5f}")

## 3. CatBoost

In [ ]:
catboost_results = cross_validate_model(get_catboost(), X, y, folds)
np.save("models/catboost_oof.npy", catboost_results["oof_preds"])
catboost_full = train_full(get_catboost(), X, y)
np.save("models/catboost_test.npy", catboost_full.predict_proba(X_test)[:,1])
print(f"CatBoost OOF AUC: {catboost_results['oof_auc']:.5f}")

## 4. Feature Importance (LightGBM)

In [ ]:
plot_feature_importance(lgbm_full, FEATURE_COLS, top_n=25)